# EcoBot — QLoRA Fine-Tuning Notebook
**Base model:** `meta-llama/Meta-Llama-3-8B-Instruct`  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**Target:** Structured JSON waste classification for Indian items  
**Runtime:** Free T4 GPU on Google Colab or Kaggle  

### Setup instructions (Kaggle)
1. Open this notebook in Kaggle (New Notebook -> Import)
2. Add your Hugging Face Token as a Secret named `HF_TOKEN`
3. Enable GPU (Accelerator -> GPU T4 x2)
4. Add your uploaded dataset so `train.jsonl` and `test.jsonl` are located at `/kaggle/input/datasets/dinesh210805/ecobot-qlora-finetune-dataset/`
5. Run all cells in order
6. Download `ecobot-classifier.Q4_K_M.gguf` and `Modelfile` from the Kaggle output directory (`/kaggle/working/ecobot-classifier-gguf/`)


In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers==4.46.3 datasets peft==0.13.2 trl==0.12.1 \
    bitsandbytes==0.44.1 accelerate==1.1.1 sentencepiece protobuf \
    llama-cpp-python huggingface_hub

In [ ]:
# Cell 2 — GPU check
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3 — Configuration
import os

# ── Model ────────────────────────────────────────────────────────────────
BASE_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'
HF_TOKEN   = os.environ.get('HF_TOKEN', '')  # set in Kaggle Secrets

# ── LoRA hyperparameters ─────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                   'gate_proj', 'up_proj', 'down_proj']

# ── Training ─────────────────────────────────────────────────────────────
EPOCHS          = 3
BATCH_SIZE      = 4
GRAD_ACCUM      = 4          # effective batch = 16
LEARNING_RATE   = 2e-4
MAX_SEQ_LEN     = 1024
WARMUP_RATIO    = 0.03
LR_SCHEDULER    = 'cosine'

# ── Paths ─────────────────────────────────────────────────────────────────
TRAIN_DATA_PATH = '/kaggle/input/datasets/dinesh210805/ecobot-qlora-finetune-dataset/train.jsonl'
TEST_DATA_PATH  = '/kaggle/input/datasets/dinesh210805/ecobot-qlora-finetune-dataset/test.jsonl'
OUTPUT_DIR      = '/kaggle/working/ecobot-classifier'
GGUF_DIR        = '/kaggle/working/ecobot-classifier-gguf'

print('Config loaded.')

In [ ]:
# Cell 4 — Load base model in 4-bit
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    use_fast=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print('Model loaded.')

In [ ]:
# Cell 5 — Apply LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 6 — Load and format dataset
from datasets import load_dataset

# Load separate train and test datasets directly
train_data = load_dataset('json', data_files=TRAIN_DATA_PATH, split='train')
eval_data = load_dataset('json', data_files=TEST_DATA_PATH, split='train')

print(f'Train size: {len(train_data)}, Eval size: {len(eval_data)}')

def format_alpaca(example):
    """Convert Alpaca format to instruction-following prompt."""
    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{example['instruction']}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{example['input']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )
    return {'text': prompt}

train_data = train_data.map(format_alpaca, remove_columns=train_data.column_names)
eval_data  = eval_data.map(format_alpaca, remove_columns=eval_data.column_names)
print('Dataset formatted. Sample:')
print(train_data[0]['text'][:200])

In [ ]:
# Cell 7 — Training
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=50,
    save_strategy='epoch',
    load_best_model_at_end=True,
    optim='paged_adamw_8bit',
    group_by_length=True,
    report_to='none',
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer.train()
print('Training complete.')

In [ ]:
# Cell 8 — Save merged model (full precision for GGUF conversion)
import os
from peft import PeftModel

MERGED_DIR = '/kaggle/working/ecobot-merged'

# Merge LoRA weights into base model
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Reload in fp16 and merge
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, token=HF_TOKEN, torch_dtype=torch.float16, device_map='auto'
)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model saved to {MERGED_DIR}')

In [ ]:
# Cell 9 — Convert to GGUF (Q4_K_M) for Ollama
import subprocess, os

LLAMA_CPP_DIR = '/kaggle/working/llama.cpp'

# Clone llama.cpp (needed for conversion script)
if not os.path.exists(LLAMA_CPP_DIR):
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/ggerganov/llama.cpp', LLAMA_CPP_DIR], check=True)
    subprocess.run(['pip', 'install', '-q', '-r', f'{LLAMA_CPP_DIR}/requirements.txt'], check=True)

os.makedirs(GGUF_DIR, exist_ok=True)

# Convert to GGUF
print('Converting to GGUF...')
subprocess.run([
    'python', f'{LLAMA_CPP_DIR}/convert_hf_to_gguf.py',
    MERGED_DIR,
    '--outfile', f'{GGUF_DIR}/ecobot-classifier-f16.gguf',
    '--outtype', 'f16',
], check=True)

# Quantize to Q4_K_M
print('Quantizing to Q4_K_M...')
subprocess.run([
    'cmake', '-B', f'{LLAMA_CPP_DIR}/build', LLAMA_CPP_DIR,
    '-DCMAKE_BUILD_TYPE=Release'
], check=True)
subprocess.run(['cmake', '--build', f'{LLAMA_CPP_DIR}/build', '--config', 'Release', '-j4'], check=True)

subprocess.run([
    f'{LLAMA_CPP_DIR}/build/bin/llama-quantize',
    f'{GGUF_DIR}/ecobot-classifier-f16.gguf',
    f'{GGUF_DIR}/ecobot-classifier.Q4_K_M.gguf',
    'Q4_K_M',
], check=True)

print(f'GGUF model ready: {GGUF_DIR}/ecobot-classifier.Q4_K_M.gguf')
import os
size = os.path.getsize(f'{GGUF_DIR}/ecobot-classifier.Q4_K_M.gguf') / 1e9
print(f'File size: {size:.2f} GB')

In [ ]:
# Cell 10 — Create Modelfile for Ollama
MODELFILE = f"""FROM {GGUF_DIR}/ecobot-classifier.Q4_K_M.gguf

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER num_ctx 2048
PARAMETER stop "<|eot_id|>"

SYSTEM \"\"\"
You are EcoBot, an expert waste classification assistant for India.
Classify the given waste item and respond ONLY with valid JSON matching this schema exactly:

{{
  \"category\": \"<wet_waste|dry_waste|hazardous|e_waste|sanitary|construction|non_recyclable>\",
  \"bin_color\": \"<green|blue|red|black|grey>\",
  \"bin_label\": \"<string>\",
  \"recyclable\": <true|false>,
  \"confidence\": \"<high|medium|low>\",
  \"reason\": \"<one sentence explanation>\",
  \"preparation_steps\": [\"<step1>\", \"<step2>\"],
  \"safety_notes\": \"<string or null>\",
  \"special_facility_required\": <true|false>
}}

Return ONLY the JSON object, no markdown, no explanation.
\"\"\"
"""

with open(f'{GGUF_DIR}/Modelfile', 'w') as f:
    f.write(MODELFILE)

print('Modelfile created.')
print('\nTo register with Ollama, run:')
print(f'  ollama create ecobot-classifier -f {GGUF_DIR}/Modelfile')

In [ ]:
# Cell 11 — Quick inference test
import json

SYSTEM = """You are EcoBot, an expert waste classification assistant for India.
Classify the given waste item and respond ONLY with valid JSON."""

def test_classify(item: str):
    inputs = tokenizer(
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{SYSTEM}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"Classify this waste item: {item}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n",
        return_tensors='pt'
    ).to(merged.device)
    
    with torch.no_grad():
        outputs = merged.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        parsed = json.loads(response.strip())
        print(f'\nItem: {item}')
        print(f'Category: {parsed["category"]} | Bin: {parsed["bin_color"]} | Confidence: {parsed["confidence"]}')
    except:
        print(f'Raw output: {response}')

# Test on common Indian waste items
test_items = [
    'old newspaper',
    'used battery',
    'banana peel',
    'broken phone',
    'plastic water bottle',
    'sanitary pad',
    'expired medicines',
    'thermocol packaging',
]

for item in test_items:
    test_classify(item)

In [ ]:
# Cell 12 — Download GGUF file from Colab
# Run this to download the quantized model to your local machine
from google.colab import files
files.download(f'{GGUF_DIR}/ecobot-classifier.Q4_K_M.gguf')
files.download(f'{GGUF_DIR}/Modelfile')
print('Downloaded. Now run locally:')
print('  ollama create ecobot-classifier -f Modelfile')
print('  ollama run ecobot-classifier "classify: old battery"')

## Post-training steps

After downloading the GGUF file:

```bash
# Register with Ollama
ollama create ecobot-classifier -f Modelfile

# Test it
ollama run ecobot-classifier "Classify this waste item: old battery"

# Verify in docker-compose (CLASSIFIER_MODE=ollama)
docker compose up
```

### Training data requirements
Generate training data before running this notebook:
```bash
# From project root
python scraping/augment_dataset.py --output data/finetuning/train_augmented.jsonl --per-item 5
```
Then upload `train_augmented.jsonl` to `/content/data/` in Colab.

### Kaggle alternative
1. Create a new Kaggle notebook
2. Add dataset: upload `train_augmented.jsonl`
3. Settings → Accelerator → GPU T4 x2
4. Copy cells into Kaggle notebook (change `DATA_PATH` to `/kaggle/input/...`)